# DIT5411 — Traditional Chinese Handwriting Recognition (Kaggle)

**Before running:** Settings → Accelerator → **GPU P100**, and Settings → Internet → **On**.

Kaggle gives 30 GPU-hours/week. The dataset is downloaded inside the notebook — no upload needed.
Full run = 3 models × 3 epochs at 128px ≈ 22 GPU-hours, fits in one week.

Checkpoints go to `/kaggle/working/results/checkpoints`. Click **Save Version** during/after training to keep them as Output; later open that version → **Copy & Run Code** to resume via `--resume`.

In [ ]:
import os, sys, subprocess, glob

if not os.path.isfile('train.py'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cveg1234/DIT5411-LuiYuHin.git'], check=True)
    os.chdir('DIT5411-LuiYuHin')
subprocess.run(['git', 'pull', '-q'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('Working dir:', os.getcwd())
print('GPU devices:', gpus)
if not gpus:
    print('WARNING: no GPU detected. Set Settings -> Accelerator -> GPU P100 before the full run.')
print('Setup done.')

## Download & extract the dataset (one time, ~15 min)
684,677 images across 13,065 characters. If `data/cleaned_data` already exists (e.g. from a saved Output), this cell skips.

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), 'data', 'cleaned_data')
if not os.path.isdir(DATA_DIR) or len(os.listdir(DATA_DIR)) < 1000:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/chenkenanalytic/handwritting_data_all.git'], check=True)
    with open('all_data.zip', 'wb') as out:
        for part in sorted(glob.glob('handwritting_data_all/all_data.zip.*')):
            with open(part, 'rb') as f:
                out.write(f.read())
    subprocess.run([sys.executable, 'extract_dataset.py',
                    '--zip', 'all_data.zip', '--dest', DATA_DIR], check=True)
print('Character folders:', len(os.listdir(DATA_DIR)))

## Quick pilot (optional — 20 classes, 2 epochs)

In [ ]:
r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                    '--output-dir', 'results_pilot', '--num-classes', '20',
                    '--epochs', '2', '--models', 'A', 'B', '--no-imagenet',
                    '--steps-per-epoch', '30'])
print('pilot exit code:', r.returncode)
print(open('results_pilot/RESULTS.md', encoding='utf-8').read())

## Train models one at a time (auto-sync with GitHub)
Each cell below: (1) pulls results already published on GitHub, (2) **skips** the model if it is already there, (3) otherwise trains it for 2 epochs with `--resume`, (4) pushes the updated results back to GitHub.

This means a new session picks up exactly where GitHub left off — no manual copying. Add the `GH_TOKEN` Kaggle Secret once (below) before running. Each new model takes ~5 GPU-hours.

### One-time: add GitHub token as a Kaggle Secret
GitHub → Settings → Developer settings → Personal access tokens → **repo** scope. Then Kaggle notebook **Settings → Secrets → Add secret** named **`GH_TOKEN`** with that token. Run the cell below to verify.

In [ ]:
print('GH_TOKEN secret:', 'FOUND' if os.environ.get('GH_TOKEN') else 'MISSING — add it in Settings -> Secrets')

In [ ]:
# ---- Model A: pull -> skip-if-done -> train -> push ----
import json as _j
print('Syncing with GitHub...')
subprocess.run([sys.executable, 'github_sync.py', 'pull'])
_rp = 'results/results.json'
_done = []
if os.path.isfile(_rp):
    _done = [m['model'] for m in _j.load(open(_rp, encoding='utf-8')).get('models', [])]
if 'A' in _done:
    print('Model A already published on GitHub — skipping training.')
else:
    print('Training Model A (2 epochs, ~5h)...')
    r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                        '--output-dir', 'results', '--epochs', '2',
                        '--models', 'A', '--resume'])
    print('Model A exit code:', r.returncode)
    if os.environ.get('GH_TOKEN'):
        subprocess.run([sys.executable, 'github_sync.py', 'push'])
    else:
        print('GH_TOKEN missing — results saved locally; add secret and re-run push.')
print(open('results/RESULTS.md', encoding='utf-8').read() if os.path.isfile('results/RESULTS.md') else 'RESULTS.md not found')

In [ ]:
# ---- Model B: pull -> skip-if-done -> train -> push ----
import json as _j
print('Syncing with GitHub...')
subprocess.run([sys.executable, 'github_sync.py', 'pull'])
_rp = 'results/results.json'
_done = []
if os.path.isfile(_rp):
    _done = [m['model'] for m in _j.load(open(_rp, encoding='utf-8')).get('models', [])]
if 'B' in _done:
    print('Model B already published on GitHub — skipping training.')
else:
    print('Training Model B (2 epochs, ~5h)...')
    r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                        '--output-dir', 'results', '--epochs', '2',
                        '--models', 'B', '--resume'])
    print('Model B exit code:', r.returncode)
    if os.environ.get('GH_TOKEN'):
        subprocess.run([sys.executable, 'github_sync.py', 'push'])
    else:
        print('GH_TOKEN missing — results saved locally; add secret and re-run push.')
print(open('results/RESULTS.md', encoding='utf-8').read() if os.path.isfile('results/RESULTS.md') else 'RESULTS.md not found')

In [ ]:
# ---- Model C: pull -> skip-if-done -> train -> push ----
import json as _j
print('Syncing with GitHub...')
subprocess.run([sys.executable, 'github_sync.py', 'pull'])
_rp = 'results/results.json'
_done = []
if os.path.isfile(_rp):
    _done = [m['model'] for m in _j.load(open(_rp, encoding='utf-8')).get('models', [])]
if 'C' in _done:
    print('Model C already published on GitHub — skipping training.')
else:
    print('Training Model C (2 epochs, ~5h)...')
    r = subprocess.run([sys.executable, 'train.py', '--data-dir', DATA_DIR,
                        '--output-dir', 'results', '--epochs', '2',
                        '--models', 'C', '--resume'])
    print('Model C exit code:', r.returncode)
    if os.environ.get('GH_TOKEN'):
        subprocess.run([sys.executable, 'github_sync.py', 'push'])
    else:
        print('GH_TOKEN missing — results saved locally; add secret and re-run push.')
print(open('results/RESULTS.md', encoding='utf-8').read() if os.path.isfile('results/RESULTS.md') else 'RESULTS.md not found')

## Final results (after A, B and C are done)

In [ ]:
from IPython.display import Markdown, display
p = 'results/RESULTS.md'
display(Markdown(open(p, encoding='utf-8').read()) if os.path.isfile(p)
        else 'RESULTS.md not generated yet — finish all models first.')